In [5]:
import pandas as pd
import sqlite3
from scipy import stats #hipotez testleri için bu kütüphaneyi kullanıcaz 
#temizlenmiş verime ulaşıcam
conn=sqlite3.connect('ifood_analiz.db')
#temizlenmiş veriyi sql den cekip hafizaya(ram) df olarak yüklüyoruz
df=pd.read_sql_query("select *from final_marketing_data",conn)
print(f"bağlantı basarıyla kuruldu!{len(df)} satir veri analizine hazir")

bağlantı basarıyla kuruldu!2211 satir veri analizine hazir


In [8]:
#ilk hipotez *bilimsel testimizle başlayalım 
#H1:müşterinin yıllık geliri(ıncome) arttıkca şarap harcaması tutarı lineer olarak artar mı?, bu iki sürekli sayısal değer arsındaki ilişkinın gücünü ,en önemlisi istatistiksel olarak anlamlı olup olmadığını (p-value<0.05) ölçmek için pearson korelasyon testi kullanılır.
#verideki olası boş değerlere karşı temiz bir kopya ile başlıycaz
df_clean=df[['Income','MntWines']].dropna()
#pearson korelasyom k.*r ve *p degerini hesaplıycaz
r_stat,p_val=stats.pearsonr(df_clean['Income'],df_clean['MntWines'])
print("---H1:gelir ve şarap harcaması ilişkisi---")
print(f"korelayon katsayısı (r):{r_stat:.4f} ")
print(f"P-değeri(anlamlılık):{p_val}")
#istatistiksel yorum
if p_val<0.05:print("/sonuç:H0 red edildi! gelir ile şarap harcaması arasında anlamlı bir ilişki vardır.")
if r_stat>0.05:print(f"Pozitif yönlü bir ilişki var.Gelir arttıkca şarap harcamasıda artiyor(r={r_stat:.2f}).")
else:print("/nsonuç:H0 red edilemei.Aralarındaki ilşki istatistiksel olarak anlmlı değildir.")
#sonuçları yorumlayalım:
#korelasyon katsayımız r=0.6882 ,bu değr bize gelir ile şarap harcaması arasında pozitif yönlü ve oldukca güçlü bir ilişki olduğunu söylüyor *0.70 e cok yakın yani tahminimiz doğru;gelir arttıkca şarap harcaması cok net bir şekilde yukarı tırmanıyor.
#p-değeri:e-310 sıfırdan sonra 310 tane sıfır var demektir*0.00000....33 bizim sınırımız ise 0.5 idi yanı bu değer 0.05 den okadar küçükki bu iki değişken arasındakı ilişkinin tesadüfen oluşma ihtimali matematiksel olarak imkansıza yakın.*H0 hipotezini masaya vurarak red edelim.


---H1:gelir ve şarap harcaması ilişkisi---
korelayon katsayısı (r):0.6882 
P-değeri(anlamlılık):3.3222607559065e-310
/sonuç:H0 red edildi! gelir ile şarap harcaması arasında anlamlı bir ilişki vardır.
Pozitif yönlü bir ilişki var.Gelir arttıkca şarap harcamasıda artiyor(r=0.69).


In [18]:
#H2:evde genç ,çocuk olmasının şarap ,et harcamasına etkisi(T-testi)
#1. evlerdeki toplam cocuk sayısını bulup çocuk var =1 yok=0 grubu oluşturalım.
#df['total_children']=df['Kidhome']+df['Teenhome']
#df['Has_child']=df['total_children'].apply(lambda x:1 if x>0 else 0)
#lüks tüketim harcaması toplamı
#cocuklu_aileler=df[df['Has_child']==1] ['Luxury_Spending'].dropna()
#cocuksuz_aileler=df[df['Has_child']==0] ['Luxury_Spending'].dropna()
#df['Luxury_Spending']=df['MntWines']+df['MntMeatProducts']
#2. Grupları ayıralım
#cocuklu_aileler=df[df['Has_child']==1] ['Luxury_Spending'].dropna()
#3.bağimsiz iki örnelkem bağımsiz T-testi
#t_stat,p_val_t=stats.ttest_ind(cocuksuz_aileler,cocuklu_aileler,equal_var=False)
#print("---H2:evde çocuk olmasi ve lüks tüketim ilişkisi---")
#print(f"Çocuksuz ailelerin ort lüks ürün harcamasi:{cocuksuz_aileler.mean():.2f}")
#print(f"Çocuklu ailelerin ort lüks harcamasi:{cocuklu_aileler.mean():.2f}")
#rint(f"T-istatistiği:{t_stat:.4f}")
#print(f"P-Değeri:{p_val_t}")
#if p_val_t<0.05:
 #   print("/nSonuc:H0 rededildi ! evde cocuk olmasi lüks tüketim harcamalari anlamli derecede etkiliyor.")
#else:
 #   print("/nSonuc:H0 reddedilemez! Çocuk varliği ile lüks küketim arasinda ilişki fark istatistiksel olarak anlamli değildir.")

In [9]:
# H2 Hipotezi: Evde çocuk/genç olmasının Şarap ve Et harcamasına etkisi (T-Testi)

# 1. Evdeki toplam çocuk ve genç sayısını bulup, "Çocuk Var(1) / Yok(0)" grubu oluşturalım
df['Total_Children'] = df['Kidhome'] + df['Teenhome']
df['Has_Child'] = df['Total_Children'].apply(lambda x: 1 if x > 0 else 0)
#lambda olsun bitsin fonk için kullanılır burda total_children daki her bir satiri kontrol eder eğer çoçok sayısı 0 dan buyukse cocuk var yazar,değilse 0 çoçuk yok yazar.
# Lüks tüketim harcamasını toplayalım (Şarap + Et)
df['Luxury_Spending'] = df['MntWines'] + df['MntMeatProducts']

# 2. Grupları ayıralım
#boş NAN değerlerini dropna ile temizleyelim
cocuklu_aileler = df[df['Has_Child'] == 1]['Luxury_Spending'].dropna()
cocuksuz_aileler = df[df['Has_Child'] == 0]['Luxury_Spending'].dropna()

# 3. Bağımsız İki Örneklem T-Testi uygulayalım
t_stat, p_val_t = stats.ttest_ind(cocuksuz_aileler, cocuklu_aileler, equal_var=False)
#equal_var)false de iki grubun varyanslarının eşit olmadığını varsayıyoruz(welch's t-test)daha güvenli bir yaklasımdır.
#:.2f virgülden sonra 2 basamak al der
print("--- H2: Evde Çocuk Olması ve Lüks Tüketim İlişkisi ---")
print(f"Çocuksuz Ailelerin Ortalama Lüks Harcaması: {cocuksuz_aileler.mean():.2f}")
print(f"Çocuklu Ailelerin Ortalama Lüks Harcaması: {cocuklu_aileler.mean():.2f}")
print(f"T-İstatistiği: {t_stat:.4f}")
print(f"P-Değeri: {p_val_t}")

if p_val_t < 0.05:
    print("\nSonuç: H0 Reddedildi! Evde çocuk olması lüks tüketim harcamalarını anlamlı derecede etkiliyor.")
else:
    print("\nSonuç: H0 Reddedilemedi. Çocuk varlığı ile lüks tüketim arasındaki fark istatistiksel olarak anlamlı değil.")
    #cıktı yorumu :Evinde çocuk veya genç olan aileler, çocuksuz ailelere kıyasla lüks tüketime (şarap ve et) neredeyse 3 kat daha az bütçe ayırıyor. 
    # Şirket eğer lüks ürün kampanyası yapacaksa, hedef kitleden çocuklu aileleri doğrudan elemesi bütçe optimizasyonu sağlar

--- H2: Evde Çocuk Olması ve Lüks Tüketim İlişkisi ---
Çocuksuz Ailelerin Ortalama Lüks Harcaması: 859.33
Çocuklu Ailelerin Ortalama Lüks Harcaması: 317.96
T-İstatistiği: 22.3152
P-Değeri: 1.8904474065850152e-87

Sonuç: H0 Reddedildi! Evde çocuk olması lüks tüketim harcamalarını anlamlı derecede etkiliyor.


In [ ]:
# H3 Hipotezi: Eğitim Seviyesi (PhD vs Graduate) ve Kampanyaya Yanıt İlişkisi (Ki-Kare Testi)

# 1. Önce sadece PhD ve Lisans (Graduation) gruplarını filtreleyelim
df_edu = df[df['Education'].isin(['PhD', 'Graduation'])]
#*isin bir secim sepetidir!!
#bu satırda ;kod öncelikle df[education]  daki tüm müşeri eğitim durumlarının yazıldığı sutüna odaklanır
#.isin(['PhD', 'Graduation']) sütundaki tüm satırları tek tek gezer istenen eğitim durumlarını secer isin=var mı? dır,ever varsa pandas evet der en diştakıdf[] evetleri (true) toplar.
# 2. İki kategorik değişken için bir Çapraz Tablo (Contingency Table) oluşturalım
cross_tab = pd.crosstab(df_edu['Education'], df_edu['Response'])
#pd.crosstab 2 değişkenin frekanslarını(sayılarını) kesiştirerek matrıs tablosu oluşturu.
# 3. Ki-Kare Testini uygulayalım
chi2, p_val_chi, dof, expected = stats.chi2_contingency(cross_tab)
#ki kare,ist. anlamlılık,serbestlik derecesi,expected ise değişkenler arası ilşki olmazsa göreceğimiz sayıların tablosu 
print("--- H3: Eğitim Seviyesi ve Kampanya Yanıt İlişkisi ---")
print("Frekans Tablosu (Mevcut Durum):")
print(cross_tab)
print(f"\nKi-Kare İstatistiği: {chi2:.4f}")
print(f"P-Değeri (Anlamlılık): {p_val_chi}")

if p_val_chi < 0.05:
    print("\nSonuç: H0 reddedildi! Eğitim seviyesi ile kampanyaya yanıt verme oranı arasında anlamlı bir ilişki vardır.")
else:
    print("\nSonuç: H0 reddedilemez! Eğitim seviyesi ile kampanyaya yanıt verme arasındaki fark istatistiksel olarak anlamlı değil.")
     #Doktora (PhD) seviyesindeki müşterilerin kampanyaya geri dönüş (oran olarak) lisans mezunlarına göre çok daha yüksek.
    #Şirket, doktora mezunlarına özel veya akademik tonlu kampanyalar kurgularsa bütçesini çok daha verimli yönetebilir

--- H3: Eğitim Seviyesi ve Kampanya Yanıt İlişkisi ---
Frekans Tablosu (Mevcut Durum):
Response      0    1
Education           
Graduation  963  152
PhD         378  101

Ki-Kare İstatistiği: 13.3869
P-Değeri (Anlamlılık): 0.00025338567851978096

Sonuç: H0 reddedildi! Eğitim seviyesi ile kampanyaya yanıt verme oranı arasında anlamlı bir ilişki vardır.


In [ ]:
# H4 Hipotezi: Müşteri Yaşı ile Mağaza Alışverişi Arasındaki Pearson Korelasyonu

# Veri setinde doğrudan 'Age' (Yaş) sütununun olup olmadığını kontrol ederek ilerleyelim. 
# Eğer yoksa Year_Birth üzerinden yaş hesabı yapıyoruz (Mevcut yıl: 2026)
if 'Age' not in df.columns and 'Year_Birth' in df.columns:
    df['Age'] = 2026 - df['Year_Birth']

# Boş değerleri uçurup temiz bir kopyayla çalışalım
df_age_store = df[['Age', 'NumStorePurchases']].dropna()

# Pearson Korelasyon Testi
r_stat_h4, p_val_h4 = stats.pearsonr(df_age_store['Age'], df_age_store['NumStorePurchases'])

print("--- H4: Yaş ve Mağaza Alışverişi İlişkisi ---")
print(f"Pearson Korelasyon Katsayısı (r): {r_stat_h4:.4f}")
print(f"P-Değeri (Anlamlılık): {p_val_h4}")
#h4 eklentim h1 ile karışmaması için hipotez 4 h4 dur.
if p_val_h4 < 0.05:
    print("\nSonuç: H0 reddedildi! Yaş ile mağazadan alışveriş yapma sayısı arasında anlamlı bir ilişki vardır.")
    if r_stat_h4 > 0:
        print(f"Pozitif yönlü bir ilişki var: Yaş arttıkça mağazadan alışveriş yapma sayısı da artıyor (r = {r_stat_h4:.2f}).")
    else:
        print(f"Negatif yönlü bir ilişki var: Yaş arttıkça mağazadan Alışveriş yapma sayısı azalıyor (r = {r_stat_h4:.2f}).")
else:
    print("\nSonuç: H0 reddedilemez! Yaş ile mağaza alışverişi arasındaki ilişki istatistiksel olarak anlamlı değil.")

--- H4: Yaş ve Mağaza Alışverişi İlişkisi ---
Pearson Korelasyon Katsayısı (r): 0.1398
P-Değeri (Anlamlılık): 4.019760492022675e-11

Sonuç: H0 reddedildi! Yaş ile mağazadan alışveriş yapma sayısı arasında anlamlı bir ilişki vardır.
Pozitif yönlü bir ilişki var: Yaş arttıkça mağazadan alışveriş yapma sayısı da artıyor (r = 0.14).


Peki buraya kadar ilerledim ama aklıma takılan bir şey var keşif analizi sırasında takıldığım ağ ,onu hala bir sonuca ulaştırmadım .Şikayet eden müşteriler hatta eğitim seviyesi yükseldikce mağazaya çüzdanını kapatan küskünler ,bu müşterilerin şikayet ettiği ortak sorun ne .Eğer bunu bulursak bu sorunu halledip müşteri memnuniyetini elde edebiliz!!

In [13]:
#hipotez 6 kök neden analizi
# Şikayet Analizi: Şikayet eden müşteriler ile etmeyenlerin web sitesi ziyareti ve harcama alışkanlıkları

print("--- ŞİKAYETÇİ MÜŞTERİLERİN PROFİLİ ---")

# 1. Toplam kaç kişi şikayet etmiş?
complain_counts = df['Complain'].value_counts()
print(f"Toplam Şikayet Etmeyen Müşteri: {complain_counts[0]}")
print(f"Toplam Şikayet Eden Müşteri: {complain_counts.get(1, 0)}\n")

# 2. Şikayet edenlerin web sitesini ziyaret sıklığı daha mı yüksek?
# (Web sitesinde teknik bir sorun mu var?)
web_vis_complain = df.groupby('Complain')['NumWebVisitsMonth'].mean()
print("Gruplara Göre Aylık Web Sitesi Ziyaret Ortalaması:")
print(f"Şikayet Etmeyenler: {web_vis_complain[0]:.2f}")
print(f"Şikayet Edenler: {web_vis_complain.get(1, 0):.2f}\n")

# 3. Şikayet edenler en çok hangi kanaldan alışveriş yapıyor?
print("Şikayet Edenlerin Alışveriş Kanalı Ortalamaları:")
channels = ['NumWebPurchases', 'NumStorePurchases', 'NumCatalogPurchases']
print(df.groupby('Complain')[channels].mean().T)
#cıktı yorumu şikayet edenler websitesini daha sık ziyaret ediyor ama alış veriş kısmında mağazadan alıyorlar gibi duruyor .sorun web site de olabılır mı?


# Ürün Bazlı Şikayet Analizi: Şikayet edenler en çok hangi ürün grubuna para harcıyor?

print("---  ÜRÜN KATEGORİSİ BAZLI ŞİKAYET ANALİZİ ---")

# Veri setindeki ürün harcamaları sütunlarını otomatik yakalayalım (Mnt ile başlayanlar)
products = [col for col in df.columns if 'Mnt' in col and 'Spending' not in col]
#kısa yol(öğrenilen * yeni kod ) ürunin basinda mnt vardı bunu kullanıcaz ama daha önce benim ürettiğim luxury_spending  gibi toplam sütunları eklemesin diye not dedik.
# Şikayet durumuna göre ürün harcama ortalamaları
products_df = df.groupby('Complain')[products].mean().T
print("Şikayet Durumuna Göre Ürün Harcama Ortalamaları:")
print(products_df)

# Ekstra ipucu: Şikayet edenlerin şikayet etmeyenlere göre harcama oranları (Oransal Değişim)
print("\n[Yorum]: Şikayet edenlerin harcama alışkanlıklarının farkı (%) :")
diff_percentage = ((products_df[1] - products_df[0]) / products_df[0]) * 100
# yeni komut öğrendik:diff_percentage.items() bu;Kutunun içine git, her satırdaki ürün adını alıp prod değişkenine, karşısındaki yüzde değerini alıp val değişkenine ata ve alt satırdaki kodu çalıştır der ,diff_percentage mnt ler ve değerlerini series olusturur tek basına okunmaz bu yuzden .items() eklenir.
for prod, val in diff_percentage.items():
    print(f"{prod}: %{val:.2f}")

--- ŞİKAYETÇİ MÜŞTERİLERİN PROFİLİ ---
Toplam Şikayet Etmeyen Müşteri: 2191
Toplam Şikayet Eden Müşteri: 20

Gruplara Göre Aylık Web Sitesi Ziyaret Ortalaması:
Şikayet Etmeyenler: 5.32
Şikayet Edenler: 5.85

Şikayet Edenlerin Alışveriş Kanalı Ortalamaları:
Complain                    0    1
NumWebPurchases      4.092195  3.7
NumStorePurchases    5.811045  5.4
NumCatalogPurchases  2.678229  2.1
---  ÜRÜN KATEGORİSİ BAZLI ŞİKAYET ANALİZİ ---
Şikayet Durumuna Göre Ürün Harcama Ortalamaları:
Complain                   0      1
MntWines          306.534916  176.7
MntFruits          26.352807   25.1
MntMeatProducts   167.553172  117.7
MntFishProducts    37.765860   26.7
MntSweetProducts   27.139662   18.2
MntGoldProds       44.092195   27.6

[Yorum]: Şikayet edenlerin harcama alışkanlıklarının farkı (%) :
MntWines: %-42.36
MntFruits: %-4.75
MntMeatProducts: %-29.75
MntFishProducts: %-29.30
MntSweetProducts: %-32.94
MntGoldProds: %-37.40
